# autoresearch — physics-consistent quantum-classical world models

**Mission:** prove that adding qubits to a parameter-matched classical surrogate improves simulation accuracy on The Well `turbulent_radiative_layer_2D`, via an input-conditioned Rydberg-analog block (the analog Hamiltonian adapts to the simulation state).

Branch `autoresearch/jul27` @ `0771869` · benchmark subset: 24 train / 3 valid trajectories (tcool 0.03/0.56/3.16), 64x192, 4 channels · metric: VRMSE (1-step and 8-step rollout) · fixed seed 1337 · 5-min sprints on one GB10.

## System architectures
- **none** — FNO2d trunk only (lift -> 4 spectral blocks -> proj)
- **mlp** — FNO2d + pooled-latent MLP bottleneck residual (matched params)
- **hea12** — FNO2d + 12q HEA quantum bottleneck (input-conditioned angles)
- **hea16** — FNO2d + 16q HEA quantum bottleneck (input-conditioned angles)
- **ryd12** — FNO2d + 12q Rydberg-analog block (data-driven detunings, trainable geometry)
- **ryd16** — FNO2d + 16q Rydberg-analog block (data-driven detunings, trainable geometry)
- **ryd20** — FNO2d + 20q Rydberg-analog block (data-driven detunings, trainable geometry)
- **mlp30** — FNO2d + MLP control matched to mr30 (30-output) parameter budget
- **mr30hea** — FNO2d + 30 simulated qubits: 3x10q HEA multi-register (blockade-decoupled patches)
- **mr30ryd** — FNO2d + 30 simulated qubits: 3x10q Rydberg-analog multi-register

## round1 comparison
| variant | vrmse@1 | vrmse@8 | steps | params | vram MB | status |
|---|---|---|---|---|---|---|
| hea12 | 0.4042 | 0.8763 | 6912 | 1334316 | 447.9 | ok |
| hea16 | 0.4831 | 1.0997 | 1872 | 1335244 | 1444.6 | ok |
| mlp | 0.4250 | 1.2128 | 10326 | 1335212 | 398.6 | ok |
| none | 0.4190 | 1.1426 | 10597 | 1331484 | 398.6 | ok |
| ryd16 | 0.4323 | 0.9131 | 2850 | 1335284 | 1480.6 | ok |

## round2 comparison
| variant | vrmse@1 | vrmse@8 | steps | params | vram MB | status |
|---|---|---|---|---|---|---|
| hea12f | 0.4289 | 1.0444 | 7270 | 1334316 | 423.9 | ok |
| ryd12 | 0.4199 | 1.1679 | 7772 | 1334348 | 426.1 | ok |

## round2s comparison
| variant | vrmse@1 | vrmse@8 | steps | params | vram MB | status |
|---|---|---|---|---|---|---|
| hea12 | 0.4856 | 1.3250 | 2500 | 1334316 | 423.9 | ok |
| hea16 | 0.4746 | 1.5077 | 2500 | 1335244 | 932.7 | ok |
| none | 0.4770 | 2.1022 | 2500 | 1331484 | 398.6 | ok |

## round3 comparison
| variant | vrmse@1 | vrmse@8 | steps | params | vram MB | status |
|---|---|---|---|---|---|---|
| mlp30_s1337 | 0.4123 | 0.9873 | 10123 | 1338442 | 398.7 | ok |
| mlp30_s1338 | 0.4673 | 1.4207 | 10023 | 1338442 | 398.7 | ok |
| mlp30_s1339 | 0.4219 | 0.9602 | 10093 | 1338442 | 398.7 | ok |
| mr30hea_s1337 | 0.4113 | 0.9058 | 4419 | 1338492 | 414.9 | ok |
| mr30hea_s1338 | 0.4629 | 1.2682 | 4424 | 1338492 | 414.9 | ok |
| mr30hea_s1339 | 0.4406 | 1.0165 | 4429 | 1338492 | 414.9 | ok |
| mr30ryd_s1337 | 0.4381 | 1.0859 | 5374 | 1338576 | 416.4 | ok |
| mr30ryd_s1338 | 0.4324 | 0.9418 | 5392 | 1338576 | 416.4 | ok |
| mr30ryd_s1339 | 0.4059 | 0.8360 | 5411 | 1338576 | 416.4 | ok |
| none_s1337 | 0.4507 | 1.3436 | 10297 | 1331484 | 398.6 | ok |
| none_s1338 | 0.3959 | 0.9255 | 10275 | 1331484 | 398.6 | ok |
| none_s1339 | 0.4214 | 1.0630 | 10306 | 1331484 | 398.6 | ok |

## Tree divergence map
```
baseline(none)
 ├─ mlp (classical matched control)
 ├─ hea12 ── hea16   (qubit scaling, digital ansatz)
 └─ ryd16            (analog Rydberg mechanism)
```
Advance rule: a branch survives if it beats BOTH `none` and `mlp` on vrmse@8 (rollout is the physics-consistency proxy).

## Sep 19 2026 — SoTA sweep (see `22-sota-research-sep19.md`)
| axis | finding | our position |
|---|---|---|
| insertion scaffold | zero-init gated pool→core→gate VQC in diffusion U-Nets, matched controls, null (2607.09108) | same scaffold; contribution is the analog register + industrial trunks + protocol |
| readout | trainable / input-conditioned non-local observables (ANO family, QFWP-ANO 2609.18655) | fixed ⟨Z_i⟩ → FW-A branch |
| encoding theory | Hamiltonian (detuning) encoding gives infinite-order nonlinearity + ESP from the Liouvillian gap (2607.12035) | our design; cite |
| configuration | effective rank + scrambling-onset diagnostics predict reservoir quality (2607.09445, 2608.25511) | FW-B: zero-training configuration rule |
| hardware realism | Pasqal noise-aware emulation validated on 3 QPUs (2607.28364) | FW-E noisy inference row |
| competitors | Physics-IQ i2v 45.5 (StrucPhysVideo); TokaMark 0.150 (TokaMind, weights); Sophelio S≥0.85 eligibility; Walrus median VRMSE on the Well | ladder updated; TokaMind trunk track B4b |

Tree divergence additions:
```
register (mr30ryd)
 ├─ FW-A readout: Z → Z+ZZ → diagonal-ANO → input-conditioned ANO
 ├─ FW-B configuration rule (R_eff / OTOC onset) replaces defaults
 ├─ FW-C recurrent register state across rollout steps
 └─ controls: mlp30 · gutz30 · FW-D structured classical (quaternion / phase-structured)
```


## Oct 7 2026 — Learn2Design-2026: Round 3 diagnosis, strategy slate, stratified CV (doc 35 §40w–§40y)

**Leaderboard (Round 3, mean best feasible loss, lower is better):** #1 judewallis −0.540 · #2 giannischalk −0.444 · #3 seament −0.238 · #4 actuwien −0.212 · #5 aekovalenko −0.154 · … · **ours 0.319 (#32 of 67)**. Full table: `runs/l2d/round3_analysis.json`.

**PCA-3 diagnosis.** Participants × topology scores: PC1 79 % (a general skill factor; ours −0.66 vs top 5 −2.3…−3.5). Dataset designs (good family loss < −0.1 vs plateau 0.33–0.45, `runs/l2d/family_pca.png`): the good family has more R = 1 mirrors (25.7 vs 20.3), more facing high-R pairs (cavities, 7.4 vs 5.3), twice the laser power (533 W vs 273 W), more maximum-mass suspensions. Optimised parameters are bang-bang (at bounds; tunings at multiples of 90°).

| id | strategy (recency) | evidence | test (stratified) | status |
|---|---|---|---|---|
| A | GPU acceleration: unlogged custom JIT, `lax.scan`, `vmap` chains | API allows raw callables after logging; #1 logs only ~2k evals | R1 timing, 2 topologies | queued (Spark) |
| B | modern optimizers: SF-AdamW (AlgoPerf 2024), ADOPT (NeurIPS 2024), Cautious Adam (2025), AdEMAMix (ICLR 2025), Prodigy (ICML 2024), SOAP (ICLR 2025), Lion | replaces NAAdamGD (Adam + noise) | O1 race, S6, 400 steps, paired | queued |
| C | kink-free power barrier surrogate | H1/H2 stall with large gradient at power thresholds | H3 diagnostic; ± barrier on B winner | running (merph) |
| D | population + Boltzmann selection (Urania) / Diffusion Evolution (ICLR 2025) / differentiable QD (CMA-MAEGA, SQUAD ICLR 2026) | plateau family is universal; leaders jump families | P1 vs v6 at matched evals, S6 | written |
| E | meta-RL move sampler (MetaBBO operator selection, MetaBox-v2 2025, READY 2026) | flips alone small (0.545→0.536) | uniform vs learned; prior learned on S6, tested on S12∖S6 | written |
| — | full exact-Hessian Newton | H1 0.776, H2 0.556 stalls; compute-heavy | dropped | — |

**Protocol.** Early checkpoints rank all participants well (ρ 15 min→final 0.84–0.87) but not the top 20 (0.49–0.54; 120 min 0.90–0.94) → successive halving: rung 1 = 400 steps on S6, rung 2 = 3k evals on S6, rung 3 = 9k evals (≈ 15 H100-min) on S6 then S12, compared against our official v6 15-min checkpoints. Adopt if mean Δ < −0.05 and ≥ 5/6 topologies improve.

```
v6 (transfer pool + NAAdamGD + cold draws)  0.319 official
 ├─ H1 exact-Newton TR ............ stall 0.776 (R2-1)   ✗
 ├─ H2 Newton + SR1 ............... stall 0.556 (R2-1)   ✗ (kink at power thresholds)
 │    └─ H3 + smooth barrier ...... running
 ├─ F1 bang-bang flips ............ 0.545 → 0.536        ~ (mutation operator only)
 └─ v7 candidates: A × B × C × D × E  (rungs 1–3)
```


## Oct 7 2026 (08:50) — Learn2Design rung results (doc 35 §40x–§40y)

**Budget convention:** RL-style environment steps (one forward simulation = 1 step; gradient / Hessian / batched calls charged their measured cost). Gradient race: 380 gradient steps per arm from the common warm start. Strata: R3-1, R3-8 plateau-hard; R3-6 escape-easy.

| arm (rung 1, best feasible loss) | R3-1 | R3-6 | R3-8 | vs NAAdamGD+barrier |
|---|---|---|---|---|
| start (dataset transfer) | 4.122 | 3.431 | 4.298 | |
| **NAAdamGD + smooth barrier** | **0.619** | 0.589 | 0.801 | ref |
| NAAdamGD (v6 core) | 3.330 | 0.552 | 0.743 | +0.87 (stall) |
| ADOPT / SF-AdamW / Cautious Adam / SOAP / AdEMAMix / Prodigy (+barrier) | 1.51–1.87 | 0.69–0.80 | 0.96–1.22 | +0.46 … +0.54, 0/3 wins |
| Lion + barrier | 2.352 | 1.468 | 1.347 | +1.05 |
| H3 exact-Hessian Newton + SR1 + barrier (Hessian = 30–32 grad steps) | 2.509 | 0.637 | 1.079 | +0.74 |
| evolution, value only (moves / DiffEvo / LRA-CMA / CMA) | 2.25 / 2.55 / 2.66 / 2.66 | 1.32 / 3.21 / 1.45 / 1.21 | killed (memguard) | far behind |

**Rung 2 (R3-1):** 8-chain population with Boltzmann selection + family-directed moves + REINFORCE move policy: 0.626 at 2,944 evaluations vs single noisy-NAAdamGD+barrier chain **0.420** (plateau from ~1,900; official v6 4-h final on this topology 0.416). Split budget loses from a poor start → P2 staged (descent, then branch) queued.

**PCA-3 of our output:** our converged design sits beyond the plateau centroid, opposite the good family on PC1: 10 high-R mirrors (good family 25.7), 3 cavities (7.4).

**Findings:** (1) the default penalty's kink at the power thresholds stalls gradient methods — the smooth barrier is the largest single gain; (2) exploration noise beats newer update rules at this budget; (3) value-only evolution and exact Newton lose at equal compute.

```
v6 0.319 official
 ├─ C barrier ............................ ✓ (R3-1 3.33 → 0.62)  → v7a = v6 + barrier (rung 3 running: paired v7a vs v6, 3k evals, R3-1/8/6)
 ├─ B modern optimizers .................. ✗ (0/3 each)  → 1b: same with noise (queued)
 ├─ H exact Newton (H1/H2/H3) ............ ✗
 ├─ O2 value-only evolution .............. ✗
 └─ D/E population + meta-RL moves (P1) .. ✗ single-start; P2 staged queued
```
